# BCSE 1593 – Data Science and Visualization Lab

## Practical: Data Cleaning and Preparation

In this practical, we will take a raw dataset and prepare it for further analysis.

The work will follow a simple sequence:

> **Load → inspect → find problems → clean → transform → verify**

We will work with the **Titanic** dataset, a well-known dataset used in the Kaggle *Titanic – Machine Learning from Disaster* competition.

### In this practical, we will learn to

- identify missing values
- measure missing data column-wise
- decide when to remove rows or columns
- fill missing numeric values
- fill missing categorical values
- remove duplicate records
- convert columns to suitable data types
- create useful derived columns
- clean text values when text cleaning is part of preparation
- extract useful information from a string column
- verify the cleaned dataset

The focus is not on making the data look perfect. The focus is on making sensible decisions before analysis or modelling.

# Part A – Getting Started

## 1. Importing pandas

Pandas will be our main tool for working with the dataset.

### Live coding

Import pandas using the standard alias `pd`.

In [1]:
print("Importing pandas")

# Write the import statement here
import pandas as pd

Importing pandas


## 2. Importing NumPy

NumPy is useful when we need `NaN`, numerical operations, or conditional transformations.

### Live coding

Import NumPy using the standard alias `np`.

In [2]:
print("Importing NumPy")

# Write the import statement here
import numpy as np

Importing NumPy


# Part B – Loading the Dataset

## 3. Dataset source – Titanic

The Titanic dataset is available through Kaggle's **Titanic – Machine Learning from Disaster** competition.

### Kaggle source

- Kaggle competition: https://www.kaggle.com/c/titanic
- Kaggle tutorial: https://www.kaggle.com/code/alexisbcook/titanic-tutorial

For this notebook, the CSV is loaded directly from a public raw-data URL. This avoids a manual download and keeps the practical reproducible.

### Direct CSV URL

```text
https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
```

The file contains passenger information such as class, name, sex, age, ticket, fare, cabin and port of embarkation, along with the survival outcome.

In [3]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

#read the csv data from this url into titanic
titanic = pd.read_csv(url)

#display first five rows
titanic.head()
print(titanic.head())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


## 4. First look at the data

Before changing anything, it is useful to understand what we have.

Check:

- number of rows and columns
- column names
- data types
- first few records
- last few records

In [4]:
print("Shape:")
print(titanic.shape)


print("\nColumns:")
print(titanic.columns)


print("\nData types:")
print(titanic.dtypes)


print("\nFirst five rows:")
print(titanic.head())


print("\nLast five rows:")
print(titanic.tail())



Shape:
(891, 12)

Columns:
Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

Data types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

First five rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Lain

## 5. Basic summary

`describe()` gives a quick numerical summary. It can help us notice unusual values and get a feel for the scale of the data.

In [5]:
#display numerical summary of dataset
titanic.describe(include='all')


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Dooley, Mr. Patrick",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


# Part C – Finding Missing Data

## 6. What is a missing value?

A missing value means that a value is not available in the dataset.

Pandas usually represents missing numerical or general values as `NaN`.

Missing data matters because:

- calculations may ignore missing values
- some algorithms cannot work directly with missing values
- removing too many rows can reduce the dataset
- filling values without thinking can introduce bias

There is no single rule that works for every column.

## 7. Count missing values

`isna()` identifies missing values. Adding the result column-wise gives the number of missing values in each column.

### Live coding

Find the number of missing values in every column.

In [6]:
#Calculate and display the number of missing values in each columns of dataset titanic

print(titanic.isna().sum())


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


## 8. Missing-value percentage

A count is useful, but a percentage gives more context.

For a column:

	Missing % = (Missing values / Total rows) X  100

### Live coding

Create a table containing:

- missing count
- missing percentage

Sort it from highest to lowest missing percentage.

In [7]:
#Create a table contaning missing count and missing percentage of titanic
missing_count = titanic.isna().sum()
missing_percentage = (missing_count / len(titanic)) * 100
missing_table = pd.DataFrame({'Missing Count': missing_count, 'Missing Percentage': missing_percentage})

#Sort it from highest to lowest percentage
missing_table = missing_table.sort_values(by='Missing Percentage', ascending=False)

#display it
print(missing_table)






             Missing Count  Missing Percentage
Cabin                  687           77.104377
Age                    177           19.865320
Embarked                 2            0.224467
PassengerId              0            0.000000
Name                     0            0.000000
Pclass                   0            0.000000
Survived                 0            0.000000
Sex                      0            0.000000
Parch                    0            0.000000
SibSp                    0            0.000000
Fare                     0            0.000000
Ticket                   0            0.000000


## 9. Which rows contain missing data?

Sometimes we need to inspect the actual records rather than only the totals.

### Live coding

Display the first few rows where `Age` is missing.

In [8]:
#Display first few rows where age is missing
missing_age = titanic[titanic['Age'].isna()]
print(missing_age.head())

    PassengerId  Survived  Pclass                           Name     Sex  Age  \
5             6         0       3               Moran, Mr. James    male  NaN   
17           18         1       2   Williams, Mr. Charles Eugene    male  NaN   
19           20         1       3        Masselmani, Mrs. Fatima  female  NaN   
26           27         0       3        Emir, Mr. Farred Chehab    male  NaN   
28           29         1       3  O'Dwyer, Miss. Ellen "Nellie"  female  NaN   

    SibSp  Parch  Ticket     Fare Cabin Embarked  
5       0      0  330877   8.4583   NaN        Q  
17      0      0  244373  13.0000   NaN        S  
19      0      0    2649   7.2250   NaN        C  
26      0      0    2631   7.2250   NaN        C  
28      0      0  330959   7.8792   NaN        Q  


## 10. Missing data by column

The Titanic data gives us three useful examples:

- `Age` has some missing values.
- `Cabin` has a large amount of missing data.
- `Embarked` has only a small number of missing values.

The appropriate action can therefore be different for different columns.

# Part D – Deciding What to Do with Missing Data

## 11. Option 1 – Remove rows

If only a very small number of records are incomplete and those records are not important for the analysis, removing them can be reasonable.

`dropna()` removes rows containing missing values.

### Example

Create a separate DataFrame containing only rows with a known `Embarked` value and compare length with titanic

In [9]:
#No missing embarked dataset
titanic_embarked = titanic.dropna(subset=["Embarked"])

#Original length
print(len(titanic))

#Length after removing missing
print(len(titanic_embarked))

#length after removing missing
print(len(titanic_embarked))


891
889
889


## 12. Option 2 – Fill missing numerical values

For a numerical column, common choices include:

- mean
- median
- a value based on domain knowledge

The **median** is often a useful choice when the distribution contains outliers.

For `Age`, we can inspect the median first.

In [10]:
#calculate and display median age
median_age = titanic["Age"].median()
print("Median age:", median_age)


Median age: 28.0


## 13. Fill missing `Age` values with the median

We will create a working copy instead of modifying the original dataset immediately.

This makes it easier to compare the raw and cleaned versions.

In [11]:
#create copy named titanic_clean
titanic_clean = titanic.copy()


#Fill median age
titanic_clean["Age"] = titanic_clean["Age"].fillna(median_age)



#check number of missing age values
print("Missing Age values:", titanic_clean["Age"].isna().sum())


Missing Age values: 0


## 14. Option 3 – Fill missing categorical values

For a categorical column, a common simple approach is to use the most frequent category, also called the **mode**.

For `Embarked`, first check the frequency of each category.

In [12]:
titanic_clean["Embarked"].value_counts(dropna=False)

,count
Embarked,
S,644
C,168
Q,77
NaN,2


### Live coding

Find the mode of `Embarked` and use it to fill its missing values.

In [13]:
#Find the mode of embarked
mode_embarked = titanic_clean["Embarked"].mode()[0]

#Fill missing embarked values with mode
titanic_clean["Embarked"] = titanic_clean["Embarked"].fillna(mode_embarked)

# Fill the missing Embarked values


print("Missing Embarked values:", titanic_clean["Embarked"].isna().sum())

Missing Embarked values: 0


## 15. Handling a column with a large amount of missing data

`Cabin` contains many missing values. Simply filling every missing cabin with the same cabin number would create information that was never present.

For this practical, we will keep a simpler approach: create a new indicator showing whether a cabin value was available, then remove the original `Cabin` column from the main cleaned table.

This preserves one useful piece of information without pretending that the missing cabin numbers are known.

In [14]:
titanic_clean["CabinKnown"] = titanic_clean["Cabin"].notna().astype(int)

titanic_clean = titanic_clean.drop(columns=["Cabin"])

titanic_clean[["CabinKnown"]].value_counts()

,count
CabinKnown,
0,687
1,204


## 16. Checking the result of missing-data treatment

A good cleaning step should be followed by a check.

### Live coding

Display the remaining missing values.

In [15]:
titanic_clean.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


# Part E – Removing Duplicate Records

## 17. Duplicate rows

Duplicate records can affect counts, averages and later analysis.

Use `duplicated()` to identify duplicate rows.

In [16]:
print("Number of duplicate rows:", titanic_clean.duplicated().sum())

Number of duplicate rows: 0


## 18. Remove duplicate rows

`drop_duplicates()` returns a version without exact duplicate rows.

### Live coding

Remove duplicate rows from `titanic_clean` and check the new row count.

In [17]:
before = len(titanic_clean)

#Clean duplicated rows from titanic_clean
titanic_clean = titanic_clean.drop_duplicates()


after = len(titanic_clean)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 891
Rows after: 891
Duplicates removed: 0


# Part F – Data Transformation

## 19. What is data transformation?

Data transformation means changing data into a more useful form for analysis.

Examples include:

- converting data types
- creating new variables
- combining existing columns
- grouping continuous values into categories
- changing numerical scales

The transformation should have a clear purpose.

## 20. Check and convert data types

Pandas usually detects data types automatically, but it is good practice to check them.

For example, `Pclass` represents passenger class. Treating it as a numeric value is useful for some calculations, while treating it as a category can be useful for categorical analysis.

For this lab, we will keep the original `Pclass` numeric and create additional prepared columns where useful.

In [18]:
titanic_clean.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


## 21. Creating `FamilySize`

The dataset contains:

- `SibSp` – number of siblings or spouses aboard
- `Parch` – number of parents or children aboard

We can combine them with the passenger to create a simple family-size variable:

    FamilySize = SibSp + Parch + 1

The `1` represents the passenger themselves.

In [19]:
#Create a new column familysize
titanic_clean["FamilySize"] = titanic_clean["SibSp"] + titanic_clean["Parch"] + 1

#Display first five rows with SibSp, Parch & Familysize
titanic_clean[["SibSp", "Parch", "FamilySize"]].head()


,SibSp,Parch,FamilySize
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


## 22. Creating `IsAlone`

A passenger with a family size of 1 travelled alone.

This converts a numerical idea into a simple binary feature.

In [20]:
#Create a column IsAlone for family size 1 as int type
titanic_clean["IsAlone"] = (titanic_clean["FamilySize"] == 1).astype(int)

#Display first 10 rows with FamilySize and IsAlone


titanic_clean[["FamilySize", "IsAlone"]].head(10)

,FamilySize,IsAlone
0,2,0
1,2,0
2,1,1
3,2,0
4,1,1
5,1,1
6,1,1
7,5,0
8,3,0
9,2,0


## 23. Creating `FarePerPerson`

The same ticket may represent more than one passenger. A simple preparation step is to divide the fare by the family size.

This is only a derived feature for practice; it is not a statement about the actual ticket pricing rules.

In [21]:
#A Columns Fare per person
titanic_clean["FarePerPerson"] = titanic_clean["Fare"] / titanic_clean["FamilySize"]

titanic_clean[["Fare", "FamilySize", "FarePerPerson"]].head()

,Fare,FamilySize,FarePerPerson
0,7.2500,2,3.62500
1,71.2833,2,35.64165
2,7.9250,1,7.92500
3,53.1000,2,26.55000
4,8.0500,1,8.05000


## 24. Grouping age into categories

Sometimes an exact numerical value is less convenient for a particular analysis.

We can create age groups such as:

- Child
- Teenager
- Adult
- Senior

### Live coding

Use `pd.cut()` to create an `AgeGroup` column.

In [22]:
bins = [0, 12, 19, 59, np.inf]
labels = ["Child", "Teenager", "Adult", "Senior"]

titanic_clean["AgeGroup"] = pd.cut(
    titanic_clean["Age"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

titanic_clean[["Age", "AgeGroup"]].head(15)

,Age,AgeGroup
0,22.0,Adult
1,38.0,Adult
2,26.0,Adult
3,35.0,Adult
4,35.0,Adult
5,28.0,Adult
6,54.0,Adult
7,2.0,Child
8,27.0,Adult
9,14.0,Teenager


# Part G – String Cleaning and Preparation

## 25. Why string manipulation belongs here

Text columns often contain useful information, but they may need cleaning before they can be analysed.

In this dataset, `Name` contains a person's name and title in one string. Extracting the title creates a useful structured feature.

We will also use simple string operations such as:

- `str.strip()` to remove unwanted spaces
- `str.lower()` to standardise text
- `str.split()` to separate parts of a string
- `str.extract()` to pull a pattern from text

These operations are being used as part of data preparation, not as a separate string-programming exercise.

## 26. Remove extra spaces from `Name`

Some names contain trailing spaces. `str.strip()` removes spaces at the beginning and end of each value.

In [23]:
#Remove extra spaces from name
titanic_clean["Name"] = titanic_clean["Name"].str.strip()

titanic_clean["Name"].head()

,Name
0,"Braund, Mr. Owen Harris"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,"Heikkinen, Miss. Laina"
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
4,"Allen, Mr. William Henry"


## 27. Standardise `Sex`

Text categories are easier to work with when they follow a consistent format.

We will convert the values to lowercase and remove surrounding spaces.

In [24]:
#Remove space and make sex as lowercase
titanic_clean["Sex"] = titanic_clean["Sex"].str.strip().str.lower()


titanic_clean["Sex"].value_counts(dropna=False)

,count
Sex,
male,577
female,314


## 28. Extracting a title from `Name`

The names follow a pattern similar to:

```text
Surname, Title. Given Name
```

For example:

```text
Braund, Mr. Owen Harris
```

The title is useful as a compact categorical variable.

### Live coding

Use `str.extract()` with a regular expression to create a `Title` column.

In [25]:
titanic_clean["Title"] = titanic_clean["Name"].str.extract(r",\s*([^.]*)\.", expand=False)

titanic_clean["Title"].value_counts().head(15)

,count
Title,
Mr,517
Miss,182
Mrs,125
Master,40
Dr,7
Rev,6
Col,2
Mlle,2
Major,2


## 29. Clean the extracted titles

Some titles occur very rarely. For a simple preparation step, we can group uncommon titles into `Rare`.

First, look at the title counts.

In [26]:
title_counts = titanic_clean["Title"].value_counts()
rare_titles = title_counts[title_counts < 10].index

print("Rare titles:")
print(list(rare_titles))

Rare titles:
['Dr', 'Rev', 'Col', 'Mlle', 'Major', 'Ms', 'Mme', 'Don', 'Lady', 'Sir', 'Capt', 'the Countess', 'Jonkheer']


### Group uncommon titles

Replace titles occurring fewer than 10 times with `Rare`.

In [27]:
titanic_clean["Title"] = titanic_clean["Title"].replace(rare_titles, "Rare")

titanic_clean["Title"].value_counts()

,count
Title,
Mr,517
Miss,182
Mrs,125
Master,40
Rare,27


## 30. Clean `Embarked` labels

`Embarked` contains short categorical codes. We can standardise the text and optionally map the codes to readable names.

First, make sure the text is consistent.

In [28]:
#embark to uppercase
titanic_clean["Embarked"] = titanic_clean["Embarked"].str.strip().str.upper()

titanic_clean["Embarked"].value_counts(dropna=False)

,count
Embarked,
S,646
C,168
Q,77


### Map codes to readable labels

For reporting, readable category names can be more convenient.

In [29]:
port_map = {
    "C": "Cherbourg",
    "Q": "Queenstown",
    "S": "Southampton"
}

#map embarked port codes with port names into a new columns embarkedname
titanic_clean["EmbarkedName"] = titanic_clean["Embarked"].map(port_map)

titanic_clean[["Embarked", "EmbarkedName"]].head(10)

,Embarked,EmbarkedName
0,S,Southampton
1,C,Cherbourg
2,S,Southampton
3,S,Southampton
4,S,Southampton
5,Q,Queenstown
6,S,Southampton
7,S,Southampton
8,S,Southampton
9,C,Cherbourg


# Part H – A Small Cleaning Pipeline

## 31. Putting the main steps together

Once the individual operations are clear, cleaning can be written as a sequence of steps.

The following version starts again from the raw dataset so that the process is easy to reproduce.

In [30]:
cleaned = titanic.copy()

# 1. Remove unnecessary spaces from text columns
cleaned["Name"] = cleaned["Name"].str.strip()
cleaned["Sex"] = cleaned["Sex"].str.strip().str.lower()
cleaned["Embarked"] = cleaned["Embarked"].str.strip().str.upper()

# 2. Fill numerical missing values
cleaned["Age"] = cleaned["Age"].fillna(cleaned["Age"].median())

# 3. Fill categorical missing values
cleaned["Embarked"] = cleaned["Embarked"].fillna(cleaned["Embarked"].mode()[0])

# 4. Preserve whether Cabin information was available
cleaned["CabinKnown"] = cleaned["Cabin"].notna().astype(int)
cleaned = cleaned.drop(columns=["Cabin"])

# 5. Remove exact duplicates
cleaned = cleaned.drop_duplicates().copy()

# 6. Create derived variables
cleaned["FamilySize"] = cleaned["SibSp"] + cleaned["Parch"] + 1
cleaned["IsAlone"] = (cleaned["FamilySize"] == 1).astype(int)
cleaned["FarePerPerson"] = cleaned["Fare"] / cleaned["FamilySize"]

# 7. Create age groups
cleaned["AgeGroup"] = pd.cut(
    cleaned["Age"],
    bins=[0, 12, 19, 59, np.inf],
    labels=["Child", "Teenager", "Adult", "Senior"],
    include_lowest=True
)

# 8. Extract and simplify title
cleaned["Title"] = cleaned["Name"].str.extract(r",\s*([^.]*)\.", expand=False)
title_counts = cleaned["Title"].value_counts()
rare_titles = title_counts[title_counts < 10].index
cleaned["Title"] = cleaned["Title"].replace(rare_titles, "Rare")

cleaned.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,CabinKnown,FamilySize,IsAlone,FarePerPerson,AgeGroup,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,0,2,0,3.62500,Adult,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,1,2,0,35.64165,Adult,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,0,1,1,7.92500,Adult,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,1,2,0,26.55000,Adult,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,0,1,1,8.05000,Adult,Mr


## 32. Verify the cleaned dataset

Cleaning is not finished until we check the result.

Look at:

- shape
- data types
- missing values
- duplicate rows
- a few records

In [31]:
print("Shape:", cleaned.shape)
print("\nData types:")
print(cleaned.dtypes)
print("\nMissing values:")
print(cleaned.isna().sum())
print("\nDuplicate rows:", cleaned.duplicated().sum())
print("\nSample records:")
display(cleaned.head())

Shape: (891, 17)

Data types:
PassengerId         int64
Survived            int64
Pclass              int64
Name               object
Sex                object
Age               float64
SibSp               int64
Parch               int64
Ticket             object
Fare              float64
Embarked           object
CabinKnown          int64
FamilySize          int64
IsAlone             int64
FarePerPerson     float64
AgeGroup         category
Title              object
dtype: object

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Embarked         0
CabinKnown       0
FamilySize       0
IsAlone          0
FarePerPerson    0
AgeGroup         0
Title            0
dtype: int64

Duplicate rows: 0

Sample records:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,CabinKnown,FamilySize,IsAlone,FarePerPerson,AgeGroup,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,0,2,0,3.62500,Adult,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,1,2,0,35.64165,Adult,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,0,1,1,7.92500,Adult,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,1,2,0,26.55000,Adult,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,0,1,1,8.05000,Adult,Mr


## 33. Compare raw and cleaned data

It is useful to see what changed rather than assuming that the cleaning worked.

In [32]:
comparison = pd.DataFrame({
    "Raw Rows": [len(titanic)],
    "Cleaned Rows": [len(cleaned)],
    "Raw Columns": [len(titanic.columns)],
    "Cleaned Columns": [len(cleaned.columns)],
    "Raw Missing Values": [titanic.isna().sum().sum()],
    "Cleaned Missing Values": [cleaned.isna().sum().sum()],
})

comparison

,Raw Rows,Cleaned Rows,Raw Columns,Cleaned Columns,Raw Missing Values,Cleaned Missing Values
0,891,891,12,17,866,0


# Part I – Save the Prepared Dataset

## 34. Save the cleaned data

A cleaned dataset is often saved so that the preparation work does not need to be repeated every time.

### Live coding

Save the prepared DataFrame as `titanic_cleaned.csv` and load it back to verify the file.

In [33]:
output_file = "titanic_cleaned.csv"
cleaned.to_csv(output_file, index=False)

reloaded = pd.read_csv(output_file)

print("Saved file:", output_file)
print("Reloaded shape:", reloaded.shape)
display(reloaded.head())

Saved file: titanic_cleaned.csv
Reloaded shape: (891, 17)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,CabinKnown,FamilySize,IsAlone,FarePerPerson,AgeGroup,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,0,2,0,3.62500,Adult,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,1,2,0,35.64165,Adult,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,0,1,1,7.92500,Adult,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,1,2,0,26.55000,Adult,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,0,1,1,8.05000,Adult,Mr


# Part J – Quick Checks

Before moving to analysis, ask these questions:

- Did I identify missing values before filling or removing them?
- Did I choose a reasonable treatment for each column?
- Did I avoid inventing values that were not known?
- Did I remove exact duplicates?
- Are numerical columns still numerical?
- Are categorical/text columns consistent?
- Do the new columns have a clear purpose?
- Did I check the dataset again after cleaning?

A clean dataset is not necessarily a dataset with zero missing values. Sometimes missingness itself contains useful information.

# Practice Set

The following exercises use the `cleaned` DataFrame unless another DataFrame is specified. Try to solve them without changing the original `titanic` DataFrame.

## Question 1 – Missing-value audit

Create a table showing the missing count and missing percentage for every column in `titanic`. Sort the result from highest to lowest missing percentage.

### Live coding
Write your solution in the cell below.

In [34]:
# Write your solution here
 # Calculate missing counts and missing percentages for every column in titanic
missing_count = titanic.isna().sum()
missing_percentage = (missing_count / len(titanic)) * 100

# Create the summary table and sort by missing percentage in descending order
q1_table = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing Percentage': missing_percentage
})
q1_table = q1_table.sort_values(by='Missing Percentage', ascending=False)

# Display the resulting table
print(q1_table)

             Missing Count  Missing Percentage
Cabin                  687           77.104377
Age                    177           19.865320
Embarked                 2            0.224467
PassengerId              0            0.000000
Name                     0            0.000000
Pclass                   0            0.000000
Survived                 0            0.000000
Sex                      0            0.000000
Parch                    0            0.000000
SibSp                    0            0.000000
Fare                     0            0.000000
Ticket                   0            0.000000


## Question 2 – Missing Age

Create a copy of `titanic` called `q2`. Fill missing `Age` values with the median age. Verify that no missing age values remain.

### Live coding
Write your solution in the cell below.

In [35]:
# Write your solution here
# Create a copy of titanic called q2
q2 = titanic.copy()

# Fill missing Age values with the median age
median_age = q2["Age"].median()
q2["Age"] = q2["Age"].fillna(median_age)

# Verify that no missing age values remain
print("Missing Age values remaining:", q2["Age"].isna().sum())

Missing Age values remaining: 0


## Question 3 – Categorical missing data

In a new DataFrame, fill missing `Embarked` values using the mode. Display the rows that originally had missing `Embarked` and show their new values.

### Live coding
Write your solution in the cell below.

In [ ]:
# Write your solution here
# Create a copy of titanic called q3
q3 = titanic.copy()

# Identify the rows that originally had missing Embarked values
originally_missing = q3[q3["Embarked"].isna()].index

# Find the mode of Embarked and fill missing values
mode_embarked = q3["Embarked"].mode()[0]
q3["Embarked"] = q3["Embarked"].fillna(mode_embarked)

# Display the rows that originally had missing Embarked and show their new values
print("Rows that originally had missing Embarked values and their new values:")
display(q3.loc[originally_missing, ["Name", "Embarked"]])

## Question 4 – Compare imputation methods

Create two copies of the Titanic data. Fill missing `Age` with the mean in one and the median in the other. Compare the two replacement values and write one or two observations as comments.

### Live coding
Write your solution in the cell below.

In [ ]:
# Write your solution here
# Write your solution here
q4_mean = titanic.copy()
q4_median = titanic.copy()

q4_mean["Age"] = q4_mean["Age"].fillna(q4_mean["Age"].mean())
q4_median["Age"] = q4_median["Age"].fillna(q4_median["Age"].median())

print("Mean Age:")
print(q4_mean["Age"].describe())

print("\nMedian Age:")
print(q4_median["Age"].describe())

# Observations: Mean is slightly higher than median due to right-skewed age distribution; median is more robust against outliers.

## Question 5 – Duplicate check

Check whether the raw Titanic dataset contains exact duplicate rows. Then create a duplicate manually and use `drop_duplicates()` to remove it.

### Live coding
Write your solution in the cell below.

In [37]:
# Write your solution here
# Check whether the raw Titanic dataset contains exact duplicate rows
duplicate_count = titanic.duplicated().sum()
print(f"Number of exact duplicate rows in raw titanic: {duplicate_count}")

# Create a duplicate manually (e.g., duplicate the first row and append it)
manual_duplicate = titanic.iloc[[0]].copy()
titanic_with_dup = pd.concat([titanic, manual_duplicate], ignore_index=True)

print(f"Rows after adding manual duplicate: {len(titanic_with_dup)}")

# Use drop_duplicates() to remove the duplicate
titanic_cleaned_dup = titanic_with_dup.drop_duplicates()

print(f"Rows after drop_duplicates(): {len(titanic_cleaned_dup)}")

Number of exact duplicate rows in raw titanic: 0
Rows after adding manual duplicate: 892
Rows after drop_duplicates(): 891


## Question 6 – Family categories

Using `FamilySize` in `cleaned`, create a new column called `FamilyType` with three categories: `Alone` for 1, `Small` for 2–4, and `Large` for more than 4.

### Live coding
Write your solution in the cell below.

In [38]:
# Write your solution here
# Create a new column called FamilyType using pd.cut() or np.select()
bins = [0, 1, 4, np.inf]
labels = ["Alone", "Small", "Large"]

cleaned["FamilyType"] = pd.cut(
    cleaned["FamilySize"],
    bins=bins,
    labels=labels,
    right=True
)

# Display the first few rows with FamilySize and FamilyType to verify
display(cleaned[["FamilySize", "FamilyType"]].head(10))

,FamilySize,FamilyType
0,2,Small
1,2,Small
2,1,Alone
3,2,Small
4,1,Alone
5,1,Alone
6,1,Alone
7,5,Large
8,3,Small
9,2,Small


## Question 7 – Fare transformation

Create a column called `FareRounded` by rounding `Fare` to two decimal places. Display `Fare` and `FareRounded` together.

### Live coding
Write your solution in the cell below.

In [39]:
# Write your solution here
# Create a column called FareRounded by rounding Fare to two decimal places
cleaned["FareRounded"] = cleaned["Fare"].round(2)

# Display Fare and FareRounded together
display(cleaned[["Fare", "FareRounded"]].head(10))

,Fare,FareRounded
0,7.2500,7.25
1,71.2833,71.28
2,7.9250,7.92
3,53.1000,53.10
4,8.0500,8.05
5,8.4583,8.46
6,51.8625,51.86
7,21.0750,21.08
8,11.1333,11.13
9,30.0708,30.07


## Question 8 – Text cleaning

Create a new column from `Name` that contains only the surname. Use string operations rather than manually entering names.

### Live coding
Write your solution in the cell below.

In [40]:
# Write your solution here
# Create a new column from Name that contains only the surname
cleaned["Surname"] = cleaned["Name"].str.split(",").str[0].str.strip()

# Display the first few rows with Name and Surname to verify
display(cleaned[["Name", "Surname"]].head(10))

,Name,Surname
0,"Braund, Mr. Owen Harris",Braund
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Cumings
2,"Heikkinen, Miss. Laina",Heikkinen
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Futrelle
4,"Allen, Mr. William Henry",Allen
5,"Moran, Mr. James",Moran
6,"McCarthy, Mr. Timothy J",McCarthy
7,"Palsson, Master. Gosta Leonard",Palsson
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",Johnson
9,"Nasser, Mrs. Nicholas (Adele Achem)",Nasser


## Question 9 – Title analysis

Display the number of passengers in each `Title` category after rare titles have been grouped. Sort the result in descending order.

### Live coding
Write your solution in the cell below.

In [41]:
# Write your solution here
# Display the number of passengers in each Title category after rare titles have been grouped, sorted in descending order
title_analysis = cleaned["Title"].value_counts(ascending=False)
print(title_analysis)

Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64


## Question 10 – Age categories

Create a different `AgeBand` column using the groups: `Child` (0–12), `Young` (13–25), `Adult` (26–59), and `Senior` (60+).

### Live coding
Write your solution in the cell below.

In [42]:
# Write your solution here
# Define bins and labels for the new age groups
bins = [0, 12, 25, 59, np.inf]
labels = ["Child", "Young", "Adult", "Senior"]

# Create the AgeBand column using pd.cut()
cleaned["AgeBand"] = pd.cut(
    cleaned["Age"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Display the first few rows with Age and AgeBand to verify
display(cleaned[["Age", "AgeBand"]].head(15))

,Age,AgeBand
0,22.0,Young
1,38.0,Adult
2,26.0,Adult
3,35.0,Adult
4,35.0,Adult
5,28.0,Adult
6,54.0,Adult
7,2.0,Child
8,27.0,Adult
9,14.0,Young


## Question 11 – Final cleaning check

Create a compact report containing the number of rows, number of columns, total missing values and duplicate rows in `cleaned`.

### Live coding
Write your solution in the cell below.

In [43]:
# Write your solution here
# Create a compact report for the cleaned dataset
report = pd.DataFrame({
    "Rows": [len(cleaned)],
    "Columns": [len(cleaned.columns)],
    "Total Missing Values": [cleaned.isna().sum().sum()],
    "Duplicate Rows": [cleaned.duplicated().sum()]
})

# Display the compact report
display(report)

,Rows,Columns,Total Missing Values,Duplicate Rows
0,891,21,0,0


## Question 12 – Mini case study

Create `final_titanic` containing `Survived`, `Pclass`, `Sex`, `Age`, `Fare`, `FamilySize`, `IsAlone`, `AgeGroup`, `Title`, and `EmbarkedName`. Verify its shape, data types and missing values.

### Live coding
Write your solution in the cell below.

In [46]:
# Create the final_titanic DataFrame (using 'Embarked' instead of 'EmbarkedName')
final_titanic = cleaned[[
    "Survived", "Pclass", "Sex", "Age", "Fare",
    "FamilySize", "IsAlone", "AgeGroup", "Title", "Embarked"
]]

# Verify its shape, data types, and missing values
print("Shape:", final_titanic.shape)
print("\nData types:")
print(final_titanic.dtypes)
print("\nMissing values:")
print(final_titanic.isna().sum())

# Display the first few rows
display(final_titanic.head())

Shape: (891, 10)

Data types:
Survived         int64
Pclass           int64
Sex             object
Age            float64
Fare           float64
FamilySize       int64
IsAlone          int64
AgeGroup      category
Title           object
Embarked        object
dtype: object

Missing values:
Survived      0
Pclass        0
Sex           0
Age           0
Fare          0
FamilySize    0
IsAlone       0
AgeGroup      0
Title         0
Embarked      0
dtype: int64


,Survived,Pclass,Sex,Age,Fare,FamilySize,IsAlone,AgeGroup,Title,Embarked
0,0,3,male,22.0,7.2500,2,0,Adult,Mr,S
1,1,1,female,38.0,71.2833,2,0,Adult,Mrs,C
2,1,3,female,26.0,7.9250,1,1,Adult,Miss,S
3,1,1,female,35.0,53.1000,2,0,Adult,Mrs,S
4,0,3,male,35.0,8.0500,1,1,Adult,Mr,S


# Final Challenge – Prepare a Dataset for Analysis

Imagine that another student will use your prepared data tomorrow without seeing the raw file.

Create a final DataFrame that:

1. contains no exact duplicate rows
2. has a sensible treatment for missing `Age` and `Embarked`
3. does not keep the mostly-missing `Cabin` column as raw text
4. contains a `CabinKnown` indicator
5. has cleaned text values in `Sex` and `Embarked`
6. contains `FamilySize` and `IsAlone`
7. contains `AgeGroup`
8. contains a cleaned `Title`
9. contains `FarePerPerson`
10. passes a final missing-value and data-type check

### Suggested final check

Display the final DataFrame's shape, columns, data types, missing-value counts and first five rows.

In [47]:
# Build your final prepared DataFrame here.
# Use the steps from the practical and add your own checks.
# Create the final_titanic DataFrame (using 'Embarked' instead of 'EmbarkedName')
final_titanic = cleaned[[
    "Survived", "Pclass", "Sex", "Age", "Fare",
    "FamilySize", "IsAlone", "AgeGroup", "Title", "Embarked"
]]

# Verify its shape, data types, and missing values
print("Shape:", final_titanic.shape)
print("\nData types:")
print(final_titanic.dtypes)
print("\nMissing values:")
print(final_titanic.isna().sum())

# Display the first few rows
display(final_titanic.head())

Shape: (891, 10)

Data types:
Survived         int64
Pclass           int64
Sex             object
Age            float64
Fare           float64
FamilySize       int64
IsAlone          int64
AgeGroup      category
Title           object
Embarked        object
dtype: object

Missing values:
Survived      0
Pclass        0
Sex           0
Age           0
Fare          0
FamilySize    0
IsAlone       0
AgeGroup      0
Title         0
Embarked      0
dtype: int64


,Survived,Pclass,Sex,Age,Fare,FamilySize,IsAlone,AgeGroup,Title,Embarked
0,0,3,male,22.0,7.2500,2,0,Adult,Mr,S
1,1,1,female,38.0,71.2833,2,0,Adult,Mrs,C
2,1,3,female,26.0,7.9250,1,1,Adult,Miss,S
3,1,1,female,35.0,53.1000,2,0,Adult,Mrs,S
4,0,3,male,35.0,8.0500,1,1,Adult,Mr,S


# Quick Reference

### Missing data

```python
df.isna().sum()
df.dropna()
df.dropna(subset=["Column"])
df["Column"].fillna(value)
df["Column"].fillna(df["Column"].median())
df["Column"].fillna(df["Column"].mode()[0])
```

### Duplicates

```python
df.duplicated().sum()
df.drop_duplicates()
```

### Transformation

```python
df["NewColumn"] = df["A"] + df["B"]
pd.cut(df["Age"], bins=[...], labels=[...])
```

### String cleaning

```python
df["Column"].str.strip()
df["Column"].str.lower()
df["Column"].str.upper()
df["Column"].str.extract(pattern)
```

### Final inspection

```python
df.shape
df.columns
df.dtypes
df.isna().sum()
df.head()
```

> **Good preparation is a cycle: inspect → decide → transform → verify.**